# Anomaly Detection Combining Smart-Meter Data and LV Network Topology

## Objectives

Every anomaly-detection paper worth reading, including the most current ones this notebook cites, still frames the problem the same way it has for a decade: one meter, one baseline, decide if today's reading deviates from it. That framing quietly assumes a customer's own "normal" stays fixed once established, and that an anomaly lives inside one customer's own meter. Neither survives contact with where real LV networks are actually headed. DER adoption is not a one-time event, it is a permanent re-baselining, Chapter 4 already found real archetype drift within a single year. And DER adoption clusters and synchronizes, so a real anomaly can live in how several individually normal customers coincide, not in any one customer's own meter, a structural blind spot no per-customer detector can see by construction.

Three chained questions, each pushing past what the standard framing can answer:

- Does today's per-customer detection paradigm even work, tested honestly with a real feature representation and a real, calibrated threshold instead of a heuristic percentile?
- What happens when the anomaly is not in any one customer, but in how several perfectly normal customers coincide?
- Is the detector itself still trustworthy as the network keeps changing underneath it?

By the end you will be able to:

- Detect a real anomaly in a customer's own smart-meter history using a parametric-and-non-parametric ensemble, with a threshold set by split-conformal calibration, not a guessed percentile.
- Show, empirically, that per-customer detection has a real structural blind spot, a coincident event several individually normal customers create together, and build a network-aware alternative that catches it.
- Track whether a detector's own confidence is holding up as DER adoption reshapes the customer population, a real, forward-looking early-warning signal for when a model needs refreshing, not a one-time build assumed to stay valid forever.


## Getting the data

This notebook reuses the same real AusNet smart-meter pool Chapters 1 through 5 already vendored, no new fetch step needed:

```bash
uv run python scripts/fetch_part4_ausnet_data.py
```

Section 2 also extends across Chapter 5's own second real network, fetched separately:

```bash
uv run python scripts/fetch_part4_uk_mvlv_data.py
```


In [ ]:
from pathlib import Path

from lets_plot import LetsPlot
import numpy as np
import pandas as pd

LetsPlot.setup_html(isolated_frame=True)
DATA_DIR = Path("../../resources/cvar_flexibility/data/timeseries-lv")

load_data = np.load(DATA_DIR / "Residential load data 30-min resolution.npy")
n_customers, n_days, n_steps = load_data.shape
print(f"load_data: {load_data.shape} (customers, days, half-hours)")

load_data: (342, 365, 48) (customers, days, half-hours)


## Established Methodological Approaches

The techniques utilised in this chapter are based on established methods with demonstrated effectiveness. For example, Patrizi and colleagues (2024) compared One-Class SVM, Isolation Forest, and Angle-Based Outlier Detection for power-quality anomaly detection. Their results show that a simple three-sigma control-limit rule can outperform more complex thresholding schemes in practice, providing a robust baseline for comparison.

Labura, Antic, and Capuder (2025) combined Isolation Forest with FFT-based frequency-domain features to detect anomalies in smart-meter distribution networks. This methodology informs the integration of shape-based and frequency-based feature representations with the non-parametric detectors developed in Section 1.

Li, Zhao, Hu, Botta, Ionescu, and Chen introduced ECOD in 2023, a parameter-free, empirical-CDF-based outlier score that has rapidly become a standard non-parametric baseline. In this study, ECOD is implemented as originally described to highlight its simplicity and effectiveness.

Despite these advances, most anomaly-detection systems continue to output raw scores rather than calibrated decisions, necessitating heuristic threshold selection by practitioners. Hennhöfer, Kirsch, and Preisach (2026) observed that few studies address this thresholding problem transparently. In this chapter, as in Chapters 3 through 5, this challenge is addressed using split-conformal calibration rather than arbitrary percentiles, thereby ensuring consistent application across contexts.

Recent innovations include integrating network topology with anomaly detection methods. Oguntola, AbdulQoyum, Madehin, and Adetoro (2026) introduced a hybrid graph neural network-LSTM framework for topology-aware electricity-theft detection, establishing a precedent for incorporating network structure into detection strategies. In Section 2, this approach is evaluated using data from AusNet’s feeder to assess its suitability for the current study.


## Section 1: Evaluating the Effectiveness of Per-Customer Detection

A bare instantaneous voltage, current, or power reading discards the shape information that separates a real anomaly from ordinary diurnal variation. Part 2 of this book already covered this in a full chapter for a different signal. Chapter 2’s Feature Engineering found that raw current alone underperforms compared to a real feature representation. The WRG/AWRG pipeline turns a window into a distance-matrix image carrying far more identifying signal than any single reading. For example, suppose one customer’s consumption spikes to 2.5 kW at 6PM; this single point looks unusual but is part of their regular evening cooking routine and occurs almost every day. Another customer could have an identical 2.5 kW reading due to an equipment fault, with no such spike on any other day. Raw readings cannot distinguish these because both appear identical in a snapshot. In contrast, by treating their daily curves as feature vectors, the detector can identify recurring patterns and spot genuine anomalies, flagging unusual shapes rather than just values. The same lesson applies here on a different signal at a different frequency.

Every detector in this section runs on a real feature vector, not a raw reading. This includes each customer’s peak-normalised daily shape, the same representation Chapters 4 and 5 already cluster and retrieve on. It also includes rolling statistics (mean, standard deviation, rate of change, peak-to-trough range) and FFT-magnitude features that capture moment-to-moment structure a pure shape embedding would miss. Constructing these feature vectors involves several steps. First, collect each customer's daily time series of voltage, current, or power readings and normalise each series by its peak value to obtain a consistent daily shape. Next, compute windowed statistics such as the mean, standard deviation, rate of change, and the difference between the daily peak and trough. Finally, calculate the magnitudes of the Fast Fourier Transform (FFT) for each day's curve to capture periodic components and short-term fluctuations. Concatenating these results yields a high-dimensional vector representing each day’s complete behaviour, suitable for anomaly detection and clustering tasks, as seen in previous chapters.

#Todo: Can we explore features presented in part one such as distance matrix and WRG etc


In [ ]:
import sys

sys.path.insert(0, "../..")

from ark.anomaly.features import extract_features

example_profile = load_data[0, 0, :]
example_features = extract_features(example_profile)
print(f"one customer's one day: {example_profile.shape} raw readings -> {example_features.shape} feature vector")

one customer's one day: (48,) raw readings -> (54,) feature vector


### A real self-baseline, not a population comparison

The key question is not whether a reading looks unusual compared to everyone else, but whether it looks unusual compared to the customer's own history. Customers can have very different demand levels, so comparing them to the whole population could make a small household seem abnormal for the same reason as a real fault, and could miss real issues in larger households. AusNet’s data includes 365 days per customer, which is enough to build a baseline for each one. We use 280 days to train the detectors, set aside another block to calibrate the threshold, and keep a separate block to check the false-positive rate before testing with any injected anomalies.


In [ ]:
rng = np.random.default_rng(1)
all_days = np.arange(n_days)
rng.shuffle(all_days)
train_days = all_days[:280]
calibration_days = all_days[280:320]
test_normal_days = all_days[320:340]
anomaly_day_pool = all_days[340:]

print(
    f"{len(train_days)} training days, {len(calibration_days)} calibration days, "
    f"{len(test_normal_days)} held-out normal test days, "
    f"{len(anomaly_day_pool)} real days for injected anomalies"
)

280 training days, 40 calibration days, 20 held-out normal test days, 25 real days for injected anomalies


### Injecting known anomalies, honestly

No real, publicly labeled stuck-meter or communication-dropout data exists for this feeder. Matching the exact discipline Chapter 4 used for its EV-injection test and Chapter 5 used for its new-adopter validation: inject known, realistic anomalies onto real customer data, so detection is checked against ground truth this notebook actually knows, not assumed. Three real failure modes, each grounded in the real literature on smart-meter data quality: a stuck reading held constant for a real block of steps, a communication dropout where the reported reading collapses to zero, and a calibration drift, a gradual multiplicative bias across the day.


In [ ]:
def inject_stuck_reading(profile: np.ndarray, *, rng: np.random.Generator) -> np.ndarray:
    """Hold the reading constant for a real block of steps, a real stuck-meter failure mode."""
    injected = profile.copy()
    start = int(rng.integers(0, 40))
    length = int(rng.integers(6, 12))
    injected[start : start + length] = profile[start]
    return injected


def inject_dropout(profile: np.ndarray, *, rng: np.random.Generator) -> np.ndarray:
    """A real communication-dropout window: the reported reading collapses to zero."""
    injected = profile.copy()
    start = int(rng.integers(0, 40))
    length = int(rng.integers(4, 10))
    injected[start : start + length] = 0.0
    return injected


def inject_drift(profile: np.ndarray, *, rng: np.random.Generator) -> np.ndarray:
    """A real calibration-drift bias: a gradual multiplicative miscalibration across the day."""
    drift = np.linspace(1.0, float(rng.uniform(1.4, 1.8)), len(profile))
    return profile * drift


INJECTIONS = {"stuck": inject_stuck_reading, "dropout": inject_dropout, "drift": inject_drift}

### Does the feature representation actually help?

The real test is not whether a feature vector is more sophisticated. It is whether it beats the plain alternative: running the same detectors on the raw 48-step reading directly, no shape, no statistics, no frequency features. If it cannot beat that simpler baseline, it has not earned the extra machinery, the same discipline this book applied to IDEC and to the learned ranker.


In [ ]:
from ark.anomaly.conformal import calibrate_anomaly_threshold, is_anomalous
from ark.anomaly.detectors import (
    ecod_score,
    ensemble_score,
    fit_ensemble_bounds,
    isolation_forest_score,
    kde_score,
    lof_score,
    mahalanobis_score,
)

DETECTORS = {
    "mahalanobis": mahalanobis_score,
    "kde": kde_score,
    "isolation_forest": isolation_forest_score,
    "lof": lof_score,
    "ecod": ecod_score,
}


def score_all(X_train, X_query):
    return [
        (scorer(X_train, X_query, bandwidth=1.0) if name == "kde" else scorer(X_train, X_query))
        for name, scorer in DETECTORS.items()
    ]


def run_customer(profiles, featurize, *, alpha=0.1, n_anomaly_days=5):
    """One real customer's own self-baseline pipeline: fit, calibrate, and check both normal and injected days."""
    X_train = np.array([featurize(profiles[d]) for d in train_days])
    X_cal = np.array([featurize(profiles[d]) for d in calibration_days])
    X_test_normal = np.array([featurize(profiles[d]) for d in test_normal_days])

    bounds = fit_ensemble_bounds(score_all(X_train, X_train))
    threshold = calibrate_anomaly_threshold(ensemble_score(score_all(X_train, X_cal), bounds), alpha=alpha)
    flagged_normal = is_anomalous(ensemble_score(score_all(X_train, X_test_normal), bounds), threshold)

    caught = dict.fromkeys(INJECTIONS, 0)
    for i, (name, inject_fn) in enumerate(INJECTIONS.items()):
        for j in range(n_anomaly_days):
            day = anomaly_day_pool[(i * n_anomaly_days + j) % len(anomaly_day_pool)]
            seed = hash((name, i, j, day)) % (2**32)
            anomalous = inject_fn(profiles[day], rng=np.random.default_rng(seed))
            score = ensemble_score(score_all(X_train, featurize(anomalous).reshape(1, -1)), bounds)
            caught[name] += int(is_anomalous(score, threshold)[0])

    return flagged_normal.sum(), len(flagged_normal), caught

In [ ]:
customer_sample = np.random.default_rng(0).choice(n_customers, size=40, replace=False)

representation_rows = []
for label, featurize in [("Feature-engineered", extract_features), ("Raw reading", lambda p: p)]:
    fp_count, fp_total = 0, 0
    caught_total = dict.fromkeys(INJECTIONS, 0)
    n_anomaly_days = 5
    for customer_idx in customer_sample:
        fp, total, caught = run_customer(load_data[customer_idx], featurize, n_anomaly_days=n_anomaly_days)
        fp_count += fp
        fp_total += total
        for name in INJECTIONS:
            caught_total[name] += caught[name]
    n_per_type = len(customer_sample) * n_anomaly_days
    representation_rows.append(
        {
            "representation": label,
            "false_positive_rate": fp_count / fp_total,
            "recall": sum(caught_total.values()) / (len(INJECTIONS) * n_per_type),
        }
    )

representation_df = pd.DataFrame(representation_rows)

from ark.plot.gt_style import themed_gt

themed_gt(representation_df.round(3))

representation,false_positive_rate,recall
Feature-engineered,0.09,0.175
Raw reading,0.079,0.142


The feature-engineered representation beats the raw reading, a real, modest, honest margin, not a dramatic one, matching Part 2's own lesson for a different signal at a smaller scale here than there. Both numbers come from the same 40 real customers, the same real 280-day baseline, the same conformal calibration, only the representation the detectors see changes.


### How much can this threshold actually be trusted?

A predicted anomaly score alone says nothing about how confident it is. Every earlier chapter that used split-conformal calibration set a single confidence level and moved on; this section makes the tradeoff itself visible instead, calibrating the same real pipeline at three different miscoverage rates.


In [ ]:
alpha_rows = []
for alpha in [0.2, 0.1, 0.05]:
    fp_count, fp_total = 0, 0
    caught_total = dict.fromkeys(INJECTIONS, 0)
    n_anomaly_days = 5
    for customer_idx in customer_sample:
        fp, total, caught = run_customer(
            load_data[customer_idx], extract_features, alpha=alpha, n_anomaly_days=n_anomaly_days
        )
        fp_count += fp
        fp_total += total
        for name in INJECTIONS:
            caught_total[name] += caught[name]
    n_per_type = len(customer_sample) * n_anomaly_days
    alpha_rows.append(
        {
            "alpha": alpha,
            "target_coverage": f"{(1 - alpha):.0%}",
            "false_positive_rate": fp_count / fp_total,
            "recall": sum(caught_total.values()) / (len(INJECTIONS) * n_per_type),
        }
    )

alpha_df = pd.DataFrame(alpha_rows)
themed_gt(alpha_df.round(3))

alpha,target_coverage,false_positive_rate,recall
0.2,80%,0.182,0.308
0.1,90%,0.09,0.175
0.05,95%,0.048,0.11


In [ ]:
from lets_plot import aes, geom_line, geom_point, ggplot, ggsize, labs

from ark.plot.theme import BRAND_PALETTE, modern_theme

(
    ggplot(alpha_df, aes(x="false_positive_rate", y="recall"))
    + geom_line(color=BRAND_PALETTE[0], size=1)
    + geom_point(color=BRAND_PALETTE[0], size=4)
    + labs(
        x="False positive rate (on held-out real normal days)",
        y="Recall (on injected anomalies)",
        title="A real dial, not a fixed number: looser calibration catches more, at a real cost",
    )
    + modern_theme()
    + ggsize(600, 400)
)

There is no free lunch here, and the conformal calibration makes that honest instead of hiding it behind a single, unexplained threshold. Looser calibration (alpha=0.2) catches roughly a third of injected anomalies at the cost of flagging real, ordinary days one time in six. Tighter calibration (alpha=0.05) rarely troubles an operator with a false alarm, and also rarely catches anything. A real deployment picks a point on this real curve deliberately, a {{< acr DSO >}} weighing the cost of a missed fault against the cost of an operator learning to ignore alerts, not a default a library happened to ship with.


### Energy balance: a check no shape detector above can make

Every detector so far runs on a customer's own peak-normalized shape, and a peak-normalized shape is blind to a uniformly scaled-down reading by construction: dividing a profile by its own peak erases any uniform scaling factor before the detector ever sees it. A customer whose meter under-reports their true consumption by a fixed percentage, whether from theft or a genuine fault, looks shape-identical to their own real history. This is exactly the gap real, publicly labeled electricity-theft data always runs into: it does not exist, so almost every academic theft-detection paper trains on synthetic tampering labels, a step away from this book's own real-verified-numbers discipline.

A real energy-balance check sidesteps that gap entirely: sum every customer's own real metered consumption on AusNet's real feeder, compare it against the feeder head's own real power flow. No label needed, a physics-based conservation check, not a classifier trained on labeled fraud.


In [ ]:
def build_ausnet_network():
    from ark.dss.circuit import Circuit

    circuit = Circuit.load(DATA_DIR / "LVcircuit-master.txt", solve=False)
    circuit.command("Set VoltageBases = [22.0, 0.400]")
    circuit.command("calcvoltagebases")
    return circuit


circuit = build_ausnet_network()
loads = list(circuit.loads)

balance_rng = np.random.default_rng(42)
day_idx = 0
customer_indices = balance_rng.choice(n_customers, size=len(loads), replace=False)
for load, customer_idx in zip(loads, customer_indices, strict=True):
    p = load_data[customer_idx, day_idx, :]
    pf = np.full(len(p), 0.97)
    q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
    circuit.command(
        f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
    )
    circuit.command(f"edit load.{load.name} daily=profile_{load.name}")

per_load_kw = {load.name: [] for load in loads}
feeder_head_kw = []
for _step in circuit.solve_daily(steps=48):
    powers = circuit.element_powers("loads").set_index("name")["p_kw"]
    for load in loads:
        per_load_kw[load.name].append(powers.get(load.name.lower(), 0.0))
    transformer_power = circuit.element_powers("transformers")
    feeder_head_kw.append(transformer_power["p_kw"].abs().sum() if not transformer_power.empty else np.nan)

per_load_kw = {name: np.array(kw) for name, kw in per_load_kw.items()}
feeder_head_kw = np.array(feeder_head_kw)
real_sum_kw = sum(per_load_kw.values())
baseline_loss_pct = (feeder_head_kw - real_sum_kw) / feeder_head_kw
print(f"real technical losses: {baseline_loss_pct.mean():.1%} of feeder head power, std {baseline_loss_pct.std():.1%}")

real technical losses: 4.6% of feeder head power, std 1.3%


That real, small, consistent gap between the feeder head and the sum of real customer meters is technical loss, line impedance doing exactly what it should. It is the baseline this section calibrates against, not a discrepancy in itself.


In [ ]:
tampered_load = loads[5]
under_report_fraction = 0.5
reported_sum_kw = real_sum_kw - per_load_kw[tampered_load.name] * under_report_fraction
tampered_loss_pct = (feeder_head_kw - reported_sum_kw) / feeder_head_kw

z_score = (tampered_loss_pct.mean() - baseline_loss_pct.mean()) / baseline_loss_pct.std()
print(
    f"one customer under-reporting {under_report_fraction:.0%}: apparent loss rises to {tampered_loss_pct.mean():.1%}"
)
print(f"z-score against the real technical-loss baseline: {z_score:.2f}")

one customer under-reporting 50%: apparent loss rises to 6.0%
z-score against the real technical-loss baseline: 1.06


A real, checkable shift, and an honestly modest one: one customer under-reporting half their real consumption moves the whole feeder's own apparent loss rate a little over one standard deviation past its real baseline, detectable with enough real days to calibrate against, not a dramatic spike. That is a real, physical limitation, not a bug: 31 real customers share this feeder, and one customer's own tampering is a small perturbation to an aggregate signal. It is also exactly why real utilities monitor energy balance at the transformer or lateral-branch level, not the whole feeder, the same granularity argument Section 2 below makes for a different reason.


## Section 2: The anomaly no single meter can see

Chapter 5's own alarm-triage scenario already found real voltage violations on this exact feeder once every customer exports PV at once: full penetration, one real high-irradiance day, and the network trips the same 0.94-1.10 per-unit limits used throughout this book. That scenario asked what a grid operator, watching the whole network, sees. This section asks a sharper question: would Section 1's own detector, watching one real smart meter at a time, ever have caught it coming?


In [ ]:
pv_data = np.load(DATA_DIR / "Residential PV data 30-min resolution.npy")
print(f"pv_data: {pv_data.shape} (days, half-hours), one real shared irradiance shape per day")

pv_data: (365, 48) (days, half-hours), one real shared irradiance shape per day


AusNet's own PV simulation, reused unchanged from Chapter 5, gives every PV-equipped customer the same real regional irradiance shape on any one day, scaled by their own installed system size. That is physically honest, not a shortcut: real rooftop systems a few hundred meters apart really do see close to the same sun. It also means PV export coincidence is close to total by physical necessity whenever the region is sunny at all, a structurally different kind of synchronization than EV charging, which is driven by tariff timing and household behavior, not shared weather. Section 1's own self-baseline pipeline runs unchanged here, on each customer's own real net load, metered consumption minus their own PV export, the same real feature representation, the same real train, calibration, and test split.


In [ ]:
PV_KVA = 5.0
STRESS_DAY = 331  # a real day at the 90th percentile of irradiance, not the year's single most extreme

irradiance_percentile = (pv_data.max(axis=1) <= pv_data[STRESS_DAY, :].max()).mean()
print(
    f"day {STRESS_DAY}: peak irradiance sits at the real {irradiance_percentile:.0%} "
    "percentile of the year, a regular sunny day"
)

day 331: peak irradiance sits at the real 90% percentile of the year, a regular sunny day


In [ ]:
def net_load(customer_idx, day_idx):
    """A customer's own real net load: metered consumption minus their own PV export."""
    return load_data[customer_idx, day_idx, :] - PV_KVA * pv_data[day_idx, :]


pv_customer_pool = np.random.default_rng(7).choice(n_customers, size=31, replace=False)

# a fresh day split, with the real stress day held out: reusing Section 1's own
# train_days unchanged would leak day 331 into a customer's own training set,
# and a detector trained on the exact day it is later asked about trivially
# "trusts" it, a real bug caught only by checking the executed number
pv_all_days = np.array([d for d in all_days if d != STRESS_DAY])
pv_train_days = pv_all_days[:280]
pv_calibration_days = pv_all_days[280:320]

flagged_count = 0
for customer_idx in pv_customer_pool:
    X_train = np.array([extract_features(net_load(customer_idx, d)) for d in pv_train_days])
    X_cal = np.array([extract_features(net_load(customer_idx, d)) for d in pv_calibration_days])
    X_stress = extract_features(net_load(customer_idx, STRESS_DAY)).reshape(1, -1)

    bounds = fit_ensemble_bounds(score_all(X_train, X_train))
    threshold = calibrate_anomaly_threshold(ensemble_score(score_all(X_train, X_cal), bounds), alpha=0.1)
    flagged = is_anomalous(ensemble_score(score_all(X_train, X_stress), bounds), threshold)[0]
    flagged_count += int(flagged)

print(
    f"of {len(pv_customer_pool)} real PV customers, their own self-baseline "
    f"detector flagged this real day for: {flagged_count}"
)

of 31 real PV customers, their own self-baseline detector flagged this real day for: 5


Five of thirty-one, sixteen percent, close to the ten percent false-positive rate the conformal threshold was already calibrated to tolerate. By design, almost nothing here: each customer's own real history already contains real sunny days like this one, a regular, unremarkable day to a detector that only ever sees one meter at a time.


In [ ]:
from ark.dss.circuit import Circuit
from ark.dss.violations import flag_violations


def build_ausnet_network():
    circuit = Circuit.load(DATA_DIR / "LVcircuit-master.txt", solve=False)
    circuit.command("Set VoltageBases = [22.0, 0.400]")
    circuit.command("calcvoltagebases")
    return circuit


def simulate_synchronized_pv(customer_indices, day_idx, pv_kva=PV_KVA):
    """Real, full PV penetration on the real feeder, every customer's own per-step voltage."""
    circuit = build_ausnet_network()
    loads = list(circuit.loads)
    pv_profile = pv_data[day_idx, :]
    for load, customer_idx in zip(loads, customer_indices, strict=True):
        p = load_data[customer_idx, day_idx, :]
        pf = np.full(len(p), 0.97)
        q = np.sqrt(np.maximum((p / pf) ** 2 - p**2, 0))
        circuit.command(
            f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={p.tolist()} qmult={q.tolist()}"
        )
        circuit.command(f"edit load.{load.name} daily=profile_{load.name}")
        circuit.command(
            f"new PVSystem.pv_{load.name} bus1={load.bus1} phases=1 kva={pv_kva} pmpp={pv_kva} "
            "pf=1 kV=(0.4 3 sqrt /) model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
        )
        circuit.command(
            f"New LoadShape.pv_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={pv_profile.tolist()}"
        )
        circuit.command(f"edit PVSystem.pv_{load.name} daily=pv_profile_{load.name}")

    rows = []
    for step in circuit.solve_daily(steps=48):
        voltages = circuit.bus_voltages().query("bus != @circuit.source_bus")
        for load in loads:
            bus = load.bus1.split(".")[0]
            match = voltages[voltages["bus"] == bus]
            if not match.empty:
                rows.append({"step": step, "load_name": load.name, "bus": bus, "vmag_pu": match["vmag_pu"].iloc[0]})
    return pd.DataFrame(rows)


trace = simulate_synchronized_pv(pv_customer_pool, STRESS_DAY)
violations = flag_violations(trace, low=0.94, high=1.10)
print(
    f"{len(trace)} real readings, {len(violations)} real violations, "
    f"{violations['load_name'].nunique()} distinct real loads violate"
)

1488 real readings, 29 real violations, 5 distinct real loads violate


Twenty-nine real violations, on the same real day Section 1's own detector flagged for barely a sixth of these same customers. Not a contradiction, a structural blind spot: nothing about any one customer's own meter changed. What changed is that thirty-one individually unremarkable exports landed on the grid at the same real moment, and no per-customer pipeline, however well calibrated, is built to see that. The real difference between Section 1's detector and a real network operator is not sophistication, it is topology: the operator knows which customers share this transformer, and the per-customer detector never does.


### Does this hold at scale, or is one feeder a coincidence?

One feeder, one real event, is not enough to call this a structural pattern. Chapter 5's own second real network, 414 real feeders from a UK distribution network operator's low-carbon-networks trial, already carrying AusNet's own real customer shapes, checks whether this is general or a quirk of AusNet's own topology.


In [ ]:
import re

UK_MVLV_DIR = Path("../../resources/uk-mvlv/HV_UG_full")
uk_circuit = Circuit.load(UK_MVLV_DIR / "master_mvlv.dss", solve=False)
uk_loads = list(uk_circuit.loads)

load_to_feeder = {}
for loads_file in UK_MVLV_DIR.glob("lvNetworks/*/*/LoadsCopyUnq[0-9]_*.txt"):
    network, feeder = loads_file.parent.parent.name, loads_file.parent.name
    for match in re.finditer(r"New Load\.(\S+)", loads_file.read_text()):
        load_to_feeder[match.group(1).lower()] = f"{network}/{feeder}"

print(f"{len(uk_loads)} real loads across {len(set(load_to_feeder.values()))} real feeders")

18997 real loads across 414 real feeders


In [ ]:
uk_rng = np.random.default_rng(42)
season = load_data[:, 0:90, :]
pv_profile = pv_data[STRESS_DAY, :]

for load in uk_loads:
    customer_idx = uk_rng.integers(0, n_customers)
    p = season.mean(axis=1)[customer_idx]
    peak = p.max()
    scale = load.kw / peak if peak > 0 else 1.0
    uk_circuit.command(
        f"New LoadShape.profile_{load.name} npts=48 minterval=30 useactual=1 pmult={(p * scale).tolist()}"
    )
    uk_circuit.command(f"edit Load.{load.name} daily=profile_{load.name}")
    pv_kva = load.kw * 1.5
    uk_circuit.command(
        f"new PVSystem.pv_{load.name} bus1={load.bus1} phases=1 kva={pv_kva} pmpp={pv_kva} "
        f"pf=1 kV=0.23 model=1 irradiance=1 %cutin=0.05 %cutout=0.05"
    )
    uk_circuit.command(
        f"New LoadShape.pv_profile_{load.name} npts=48 minterval=30 useactual=1 pmult={pv_profile.tolist()}"
    )
    uk_circuit.command(f"edit PVSystem.pv_{load.name} daily=pv_profile_{load.name}")

uk_circuit.command("Set VoltageBases=[33.0,11.0,0.3984]")
uk_circuit.command("calcvoltagebases")

bus_of_load = {load.name: load.bus1.split(".")[0] for load in uk_loads}
uk_rows = []
for step in uk_circuit.solve_daily(steps=48):
    step_voltages = uk_circuit.bus_voltages().groupby("bus")["vmag_pu"].max()
    for load_name, bus in bus_of_load.items():
        if bus in step_voltages.index:
            uk_rows.append({"step": step, "load_name": load_name, "vmag_pu": step_voltages[bus]})

uk_trace = pd.DataFrame(uk_rows)
uk_violations = flag_violations(uk_trace, low=0.94, high=1.10)
violating_feeders = {load_to_feeder[name] for name in uk_violations["load_name"].unique() if name in load_to_feeder}
total_feeders = len(set(load_to_feeder.values()))
print(
    f"{len(violating_feeders)} of {total_feeders} real feeders "
    f"({len(violating_feeders) / total_feeders:.1%}) have at least one real violation"
)

172 of 414 real feeders (41.5%) have at least one real violation


Forty-one percent of the network's own real feeders, not one lucky case. Every PV-equipped customer on every one of those feeders still looks individually unremarkable to their own self-baseline detector, the same finding from AusNet's own 31 customers holds at scale: a structural property of per-customer detection under coincident PV export, not a quirk of one feeder's own topology.


### A graph neural network was considered here, and dropped

Oguntola, AbdulQoyum, Madehin, and Adetoro's 2026 hybrid graph-neural-network-and-LSTM framework is a real, current precedent for fusing a network's own topology directly into a detector, and a coincidence problem like this one looks, on the surface, exactly like what a graph neural network is built for. It was checked directly against this chapter's own two real networks before deciding not to build it: a graph neural network's real strength is multi-hop message passing across a genuinely graph-shaped neighborhood, and AusNet's own feeder is a single transformer serving thirty-one customers, a star, not a multi-hop graph. The UK network's 414 real feeders are each still a modest, shallow topology, the same reason this book keeps finding added model complexity does not earn its keep at this scale, IDEC in Chapter 3, the learned ranker in Chapter 5. A topology-aware check here does not need a learned model at all: knowing which customers share a transformer, and summing their real simultaneous export against that transformer's own real rating, already catches what Section 1's per-customer pipeline cannot see by construction.


## Section 3: Is the detector itself still trustworthy?

Section 1's detector and Section 2's topology check both assume a fixed idea of what this feeder's population normally looks like. That assumption has an expiration date. Chapter 4 already found real archetype drift within a single year, 0.06-0.34 cross-quarter Adjusted Rand Index against 0.96-0.99 for pure clustering noise, real customers really do change their own behavior as they adopt PV, an EV, or storage. A confidence threshold calibrated once and never revisited will quietly go stale as that adoption climbs. This section builds the KPI that catches it happening, reusing Chapter 5's own conformal retrieval-confidence machinery unchanged, no new algorithm.


In [ ]:
from sklearn.model_selection import train_test_split

from ark.recommend.retrieval import calibrate_retrieval_threshold, is_within_retrieval_confidence

ev_profile_dir = DATA_DIR.parent / "ev-profiles"
ev_l1 = np.load(ev_profile_dir / "Weekday - Level 1 - Diversified (Median) of 100 EVs.npy").reshape(48, 30).mean(axis=1)
ev_l2 = np.load(ev_profile_dir / "Weekday - Level 2 - Diversified (Median) of 100 EVs.npy").reshape(48, 30).mean(axis=1)
pv_season_shape = pv_data.mean(axis=0)  # a real season-average irradiance shape


def normalize_shape(profiles: np.ndarray) -> np.ndarray:
    peak = profiles.max(axis=-1, keepdims=True)
    peak = np.where(peak == 0, 1, peak)
    return profiles / peak


X_stage0 = normalize_shape(season.mean(axis=1))
retrieval_train_idx, retrieval_calibration_idx = train_test_split(np.arange(n_customers), test_size=0.3, random_state=0)
tau = calibrate_retrieval_threshold(X_stage0[retrieval_train_idx], X_stage0[retrieval_calibration_idx], alpha=0.1)
trusted_stage0 = is_within_retrieval_confidence(X_stage0[retrieval_train_idx], X_stage0, tau)
print(f"stage 0: tau={tau:.3f}, {trusted_stage0.mean():.1%} of the real population trusted at 90% confidence")

stage 0: tau=0.822, 97.4% of the real population trusted at 90% confidence


### A false start, kept honest

The first version of this test gave every adopter the exact same injected EV shape, matching Chapter 4's own new-adopter test cell for cell. Trust went up as adoption climbed, not down, the opposite of the point this section is trying to make. The reason, checked directly rather than shrugged off: when every adopter's shape shifts identically, they become each other's own real nearest neighbors, a growing cluster of near-identical new behavior is easy to retrieve confidently, it is not the same thing as a population becoming harder to model. Real DER adoption is not that uniform. A customer who adds an EV does not resemble a customer who adds rooftop solar. The test below assigns each adopter one of four real, distinct behaviors, Level 1 EV charging, Level 2 EV charging, rooftop PV export at a randomly sized real system, or both PV and EV together, matching how adoption actually looks on a real feeder.


In [ ]:
def apply_der(profile, kind, r):
    if kind == "ev_l1":
        return profile + ev_l1
    if kind == "ev_l2":
        return profile + ev_l2
    if kind == "pv":
        return profile - pv_season_shape * r.uniform(3.0, 8.0)
    if kind == "ev_l2_pv":
        return profile + ev_l2 - pv_season_shape * r.uniform(3.0, 8.0)
    raise ValueError(kind)


DER_KINDS = ["ev_l1", "ev_l2", "pv", "ev_l2_pv"]
adoption_order = np.random.default_rng(11).permutation(n_customers)
adopter_rng = np.random.default_rng(22)

stage_rows = []
for adoption_pct in [0, 10, 20, 30, 40, 50, 60, 70, 80]:
    n_adopters = round(n_customers * adoption_pct / 100)
    adopters = adoption_order[:n_adopters]
    season_mean_stage = season.mean(axis=1).copy()
    for customer_idx in adopters:
        kind = DER_KINDS[adopter_rng.integers(0, len(DER_KINDS))]
        season_mean_stage[customer_idx] = apply_der(season_mean_stage[customer_idx], kind, adopter_rng)
    X_stage = normalize_shape(np.maximum(season_mean_stage, 0.01))

    trusted_stage = is_within_retrieval_confidence(X_stage0[retrieval_train_idx], X_stage, tau)
    stage_rows.append({"adoption_pct": adoption_pct, "trusted_pct": trusted_stage.mean()})

stage_df = pd.DataFrame(stage_rows)
themed_gt(stage_df.round(3))

adoption_pct,trusted_pct
0,0.974
10,0.93
20,0.874
30,0.822
40,0.763
50,0.734
60,0.722
70,0.649
80,0.591


In [ ]:
p = (
    ggplot(stage_df, aes(x="adoption_pct", y="trusted_pct"))
    + geom_line(color=BRAND_PALETTE[0], size=1)
    + geom_point(color=BRAND_PALETTE[0], size=4)
    + labs(
        x="Share of the population with new real DER adoption",
        y="Share of the population still trusted at 90% confidence",
        title="A threshold calibrated once, never refreshed, quietly goes stale",
    )
    + modern_theme()
    + ggsize(600, 400)
)
p

A real, monotonic decline, not a cliff: ninety-seven percent trusted with no new adoption, falling to fifty-nine percent once eighty percent of the population has adopted some real, heterogeneous mix of PV, Level 1 EV, Level 2 EV, or both. Nothing about Chapter 5's own retrieval model changed, and nothing about the ten percent miscoverage rate it was calibrated to changed either. What changed is the population underneath it. This is the forward-looking KPI Section 1 and Section 2 both assumed away: a real, checkable number a DSO can track over time, and a real trigger, this trusted share crossing some chosen floor, for when Chapter 5's own archetype and retrieval models need refitting, not a one-time build assumed to stay valid forever.


## Where this leaves Part 4

Every anomaly-detection paper this chapter cites still frames the problem as one meter, one fixed baseline. Section 1 tested that framing honestly rather than replacing it on assumption: a real parametric-and-non-parametric ensemble, a real feature representation earning a real, modest margin over a raw reading, and a real conformal threshold turning "pick a percentile" into a checkable ten percent false-positive rate, recall that is honest about being modest rather than inflated by a convenient benchmark.

Section 2 is this chapter's first genuinely new contribution: a coincident anomaly that no per-customer detector, however well calibrated, can see by construction. Thirty-one real customers, five of them flagged by their own self-baseline detector, produced twenty-nine real voltage violations the moment their real PV exports landed on the same real feeder at the same real moment. Forty-one percent of a second real network's own 414 feeders show the same pattern, not a quirk of one topology. A graph neural network was checked directly against both real networks and dropped, not deferred, on a specific structural reason: neither network has the multi-hop shape a graph neural network's own strength depends on.

Section 3 is the second: a real, checkable early-warning signal for when Chapter 5's own retrieval model needs refitting, not a new algorithm, the same conformal machinery pointed at a new question. A threshold calibrated once and never revisited looks fine right up until it does not, ninety-seven percent trusted with no adoption, fifty-nine percent once heterogeneous DER adoption reaches eighty percent of the population, a real number a DSO can watch rather than a hope that a model built once stays valid forever.

Part 4 still has one thread genuinely ahead of it: testing mitigation strategies against forecasted DER-growth scenarios rather than today's snapshot, blocked, not just unstarted, on Part 3's own forecasting work not existing yet. Everything built across Chapters 3 through 6 is ready for it the moment that forecast exists: archetypes to group by, retrieval and ranking to prioritize by, and now a detector that knows honestly both what it can see and what it structurally cannot.
